# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method

I will treat this as a **ranking problem for content-refresh review**, not as a claim about causation or Google’s algorithm.

- **Primary method:** Logistic Regression
- **Why:** It provides an interpretable probability that a page belongs in the decline-priority group.
- **How it ranks pages:** Pages will be ordered by that predicted probability so a human reviewer can inspect the highest-priority pages first.
- **Comparison:** I will compare the model with my Week-4 baseline, which prioritizes pages with high impressions, positions 4–20, and CTR at or below 0.5%.
- **Model-selection rule:** A more complex model will only be considered useful if it improves the same evaluation metrics on the same held-out data. Complexity alone will not count as an improvement.

The result will be used as **decision support** for human review. It will not prove that a content refresh caused a traffic recovery.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Validation design

I will use a **grouped train-test split by `client_id`**. All pages from one client will stay entirely in either the training set or the test set, so the model is evaluated on clients it did not see during training.

This is appropriate because pages from the same client may share characteristics. A regular random row split could place very similar pages from one client in both sets and make the model appear stronger than it is.

I will use a fixed random seed so the result is reproducible. Before fitting any imputer, encoder, scaler, or model, I will create the split. All preprocessing decisions will be learned from the training data only.

I will verify that the training and test client groups do not overlap, then report the number of rows and the decline-label rate in each set.

In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} content items")
df.head()

Loaded 30,000 content items


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df["trend_outcome"] = [1 if x == "down" else 0 for x in df["trend_direction"]]
df["trend_outcome"].head()

0    1
1    1
2    1
3    0
4    1
Name: trend_outcome, dtype: int64

In [3]:
import plotly.express as px
outcome_counts = (
    df["trend_outcome"]
    .value_counts()
    .sort_index()
    .rename_axis("trend_outcome")
    .reset_index(name="count")
)

print(f"Trend outcome counts:\n{outcome_counts}")

fig = px.bar(
    outcome_counts,
    x="trend_outcome",
    y="count",
    labels={"trend_outcome": "Trend Outcome", "count": "Count"},
    title="Distribution of Trend Outcomes"
).show()


Trend outcome counts:
   trend_outcome  count
0              0  13738
1              1  16262


In [4]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_indices, test_indices = next(
    splitter.split(
        df,
        y=df["trend_outcome"],
        groups=df["client_id"],
    )
)

train_df = df.iloc[train_indices].copy()
test_df = df.iloc[test_indices].copy()

excluded_columns = [
    "trend_outcome",
    "trend_direction",
    "client_id",
    "content_id",
]

feature_columns = [
    column for column in df.columns
    if column not in excluded_columns
]

x_train = train_df[feature_columns]
x_test = test_df[feature_columns]

y_train = train_df["trend_outcome"]
y_test = test_df["trend_outcome"]

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])
client_overlap = train_clients.intersection(test_clients)

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Client overlap: {len(client_overlap)}")
print(f"Training decline rate: {y_train.mean():.3f}")
print(f"Test decline rate: {y_test.mean():.3f}")

assert len(client_overlap) == 0

Training rows: 23,837
Test rows: 6,163
Training clients: 25
Test clients: 7
Client overlap: 0
Training decline rate: 0.550
Test decline rate: 0.511


In [5]:
prohibited_columns = {
    "trend_outcome",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id",
    "provider_used",
    "model_used",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
}

feature_columns = [
    column for column in df.columns
    if column not in prohibited_columns
]

x_train = train_df[feature_columns]
x_test = test_df[feature_columns]

print(f"Number of model features: {len(feature_columns)}")
print("Features excluded from modeling:")
print(sorted(prohibited_columns.intersection(df.columns)))

assert "trend_outcome" not in feature_columns
assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert "client_id" not in feature_columns
assert "content_id" not in feature_columns
assert "provider_used" not in feature_columns
assert "model_used" not in feature_columns

Number of model features: 32
Features excluded from modeling:
['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'model_used', 'provider_used', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_outcome', 'trend_pct']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns = x_train.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = x_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

train_missing = x_train.isnull().sum()
test_missing = x_test.isnull().sum()

print("x_train categorical missing columns")
print(train_missing[(train_missing > 0) & train_missing.index.isin(categorical_columns)])
print("x_train numerical missing columns")
print(train_missing[(train_missing > 0) & train_missing.index.isin(numerical_columns)])

print("x_test categorical missing columns")
print(test_missing[(test_missing > 0) & test_missing.index.isin(categorical_columns)])
print("x_test numerical missing columns")
print(test_missing[(test_missing > 0) & test_missing.index.isin(numerical_columns)])

x_train categorical missing columns
competition_level    2444
main_intent          2356
word_count_tier      6614
char_count_tier      6614
dtype: int64
x_train numerical missing columns
search_volume    2319
competition      2319
cpc              2319
word_count       6614
char_count       6614
scroll_rate        15
dtype: int64
x_test categorical missing columns
competition_level     166
main_intent            18
word_count_tier      1085
char_count_tier      1085
dtype: int64
x_test numerical missing columns
search_volume     149
competition       149
cpc               149
word_count       1085
char_count       1085
scroll_rate       110
dtype: int64


In [7]:
for i in ["search_volume", "competition", "cpc", "word_count", "char_count", "scroll_rate"]:
    x_train[i].fillna(x_train[i].median(), inplace=True)
    x_test[i].fillna(x_train[i].median(), inplace=True)

for i in ["competition_level", "main_intent", "word_count_tier", "char_count_tier"]:
    x_train[i].fillna("missing", inplace=True)
    x_test[i].fillna("missing", inplace=True)

C:\Users\omara\AppData\Local\Temp\ipykernel_17128\1798364378.py:2: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



C:\Users\omara\AppData\Local\Temp\ipykernel_17128\1798364378.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\omara\AppData\Local\Temp\ipykernel_17128\1798364378.py:3: FutureWarning:

A value is trying to be set on a cop

In [8]:
standarize = StandardScaler()
x_train_scaled = standarize.fit_transform(x_train[numerical_columns])
x_test_scaled = standarize.transform(x_test[numerical_columns])

In [9]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
x_train_encoded = encoder.fit_transform(x_train[categorical_columns])
x_test_encoded = encoder.transform(x_test[categorical_columns])

In [10]:
import numpy as np

x_train_final = np.hstack([x_train_scaled, x_train_encoded])
x_test_final = np.hstack([x_test_scaled, x_test_encoded])

print("Training matrix shape:", x_train_final.shape)
print("Test matrix shape:", x_test_final.shape)

Training matrix shape: (23837, 62)
Test matrix shape: (6163, 62)


In [11]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(x_train_final, y_train)
probabilities = model.predict_proba(x_test_final)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, probabilities))
print("Average precision:", average_precision_score(y_test, probabilities))

ROC-AUC: 0.5821266396701072
Average precision: 0.5757883531424012


In [12]:
test_results = test_df[["content_id", "impressions_90d", "avg_position", "ctr"]].copy()
test_results["actual_outcome"] = y_test.values
test_results["model_probability"] = probabilities

test_results["baseline_eligible"] = (
    (test_results["impressions_90d"] >= 3000)
    & (test_results["avg_position"].between(4, 20))
    & (test_results["ctr"] <= 0.5)
)

test_results["baseline_score"] = (
    test_results["impressions_90d"]
    * (21 - test_results["avg_position"])
).where(test_results["baseline_eligible"], 0)

model_ranked = test_results.sort_values(
    "model_probability",
    ascending=False
)

baseline_ranked = test_results.sort_values(
    "baseline_score",
    ascending=False
)

for k in [10, 20, 50]:
    model_precision = model_ranked.head(k)["actual_outcome"].mean()
    baseline_precision = baseline_ranked.head(k)["actual_outcome"].mean()

    print(f"Precision@{k}")
    print(f"Model: {model_precision:.3f}")
    print(f"Baseline: {baseline_precision:.3f}")
    print()

Precision@10
Model: 0.700
Baseline: 0.200

Precision@20
Model: 0.600
Baseline: 0.400

Precision@50
Model: 0.700
Baseline: 0.400



In [13]:
comparison_table = pd.DataFrame(
    {
        "method": [
            "Logistic Regression",
            "Week-4 Baseline",
        ],
        "precision_at_10": [
            model_ranked.head(10)["actual_outcome"].mean(),
            baseline_ranked.head(10)["actual_outcome"].mean(),
        ],
        "precision_at_20": [
            model_ranked.head(20)["actual_outcome"].mean(),
            baseline_ranked.head(20)["actual_outcome"].mean(),
        ],
        "precision_at_50": [
            model_ranked.head(50)["actual_outcome"].mean(),
            baseline_ranked.head(50)["actual_outcome"].mean(),
        ],
    }
)

print(f"Test decline base rate: {y_test.mean():.3f}")
comparison_table

Test decline base rate: 0.511


,method,precision_at_10,precision_at_20,precision_at_50
0,Logistic Regression,0.7,0.6,0.7
1,Week-4 Baseline,0.2,0.4,0.4


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [14]:
test_results["model_prediction"] = (
    test_results["model_probability"] >= 0.5
).astype(int)

test_results["error_type"] = "correct"

test_results.loc[
    (test_results["actual_outcome"] == 0)
    & (test_results["model_prediction"] == 1),
    "error_type",
] = "false_positive"

test_results.loc[
    (test_results["actual_outcome"] == 1)
    & (test_results["model_prediction"] == 0),
    "error_type",
] = "false_negative"

print(test_results["error_type"].value_counts())

test_results[
    test_results["error_type"] != "correct"
][
    [
        "content_id",
        "actual_outcome",
        "model_probability",
        "model_prediction",
        "error_type",
    ]
].head(10)

test_results["content_type"] = test_df["content_type"].values

error_summary = (
    test_results
    .groupby(["content_type", "error_type"])
    .size()
    .unstack(fill_value=0)
)

error_summary

error_type
correct           3472
false_positive    1831
false_negative     860
Name: count, dtype: int64


error_type,correct,false_negative,false_positive
content_type,,,
keyword article,3472,860,1831


In [15]:
test_results["impressions_group"] = pd.cut(
    test_results["impressions_90d"],
    bins=[-1, 100, 1000, 10000, float("inf")],
    labels=["0-100", "101-1,000", "1,001-10,000", "10,000+"],
)

impressions_error_summary = (
    test_results
    .groupby(["impressions_group", "error_type"], observed=False)
    .size()
    .unstack(fill_value=0)
)

impressions_error_summary

error_type,correct,false_negative,false_positive
impressions_group,,,
0-100,1268,446,478
"101-1,000",922,219,444
"1,001-10,000",998,125,736
"10,000+",284,70,173


In [16]:
from sklearn.inspection import permutation_importance

feature_names = numerical_columns + list(
    encoder.get_feature_names_out(categorical_columns)
)

importance = permutation_importance(
    model,
    x_test_final,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
)

importance_table = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": importance.importances_mean,
    }
).sort_values("importance", ascending=False)

importance_table.head(10)

,feature,importance
13,days_with_impressions,0.045464
14,days_with_sessions,0.039126
15,content_age_days,0.029666
19,avg_position,0.027314
9,users_90d,0.021313
39,freshness_tier_0-30,0.014834
21,scroll_rate,0.009825
3,word_count,0.007919
58,position_tier_page_1,0.005358
35,age_tier_181-365,0.004654


### Error analysis and interpretation

The model produced 1,831 false positives and 860 false negatives on the held-out test set. This means it more often flagged pages as declining when the observed outcome was 0 than it missed pages whose observed outcome was 1. Several examples were close to the 0.5 classification threshold, so some errors appear to be borderline cases rather than highly confident mistakes.

The error pattern varied by impressions. False positives were highest among pages with 1,001–10,000 impressions, with 736 cases. The test set contained only one content type, `keyword article`, so the content-type comparison was limited and does not show differences between content types.

The three most important features by permutation importance were `days_with_impressions`, `days_with_sessions`, and `content_age_days`. These are plausible directional signals because they describe visibility, engagement history, and page age. They are not direct definitions of the decline label, so they do not show obvious leakage. Overall, the model is useful as a ranking aid for human review, but its errors show that the score should not be treated as proof that a page needs a specific refresh.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.